# Ejercicio 9: Uso de la API de Google Gemini

En este ejercicio vamos a aprender a utilizar la API de Google Gemini

## 1. Uso básico

El siguiente código sirve para conectarse con la API de Google Gemini de forma básica

In [2]:
!pip install -q google-genai scikit-learn numpy

In [3]:
from google import genai

CLAVE_LIMPIA = "AQ.Ab8RN6IJXa_LxrdWuCOknCg0V_diETyOKsKx_wLd7YCmSoxNZQ"
client = genai.Client(api_key=CLAVE_LIMPIA)

print("Modelos disponibles para esta API Key:\n" + "-"*40)
# Listamos todos los modelos disponibles
for m in client.models.list():
    if "flash" in m.name or "pro" in m.name:
        print(m.name)

Modelos disponibles para esta API Key:
----------------------------------------
models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-omni-flash-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/deep-research-pro-preview-12-2

In [4]:
import os
from google import genai

CLAVE_LIMPIA = "AQ.Ab8RN6IJXa_LxrdWuCOknCg0V_diETyOKsKx_wLd7YCmSoxNZQ"

# Inicializamos el cliente forzando esta nueva clave.
client = genai.Client(api_key=CLAVE_LIMPIA)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explicame en una frase qué es un modelo de lenguaje grande (LLM)."
)

print(response.text)

Es un **modelo de inteligencia artificial** entrenado con cantidades masivas de texto para **comprender, generar y responder** a lenguaje humano de forma coherente y contextual.


## 2. Retrieval

### 2.1 Cargo el corpus de 20 News Groups

In [5]:
from sklearn.datasets import fetch_20newsgroups

# Carga de un subconjunto de categorías para que el ejercicio sea más rápido
categorias = [
    "sci.space",
    "comp.graphics",
    "rec.sport.baseball",
    "talk.politics.mideast",
    "sci.med"
]

newsgroups = fetch_20newsgroups(
    subset="train",
    categories=categorias,
    remove=("headers", "footers", "quotes")
)

import random
random.seed(42)

N_DOCS = 50
indices = random.sample(range(len(newsgroups.data)), N_DOCS)

documentos = [newsgroups.data[i].strip() for i in indices]
etiquetas = [newsgroups.target_names[newsgroups.target[i]] for i in indices]

# Filtrado de documentos vacíos o demasiado cortos
documentos_filtrados = []
etiquetas_filtradas = []
for doc, et in zip(documentos, etiquetas):
    if len(doc) > 50:
        documentos_filtrados.append(doc)
        etiquetas_filtradas.append(et)

documentos = documentos_filtrados
etiquetas = etiquetas_filtradas

print(f"Cantidad de documentos cargados: {len(documentos)}")
print("\nEjemplo de documento:\n")
print(documentos[0][:500])
print(f"\nCategoría: {etiquetas[0]}")

Cantidad de documentos cargados: 47

Ejemplo de documento:

Hi ... Recently I found XV for MS-DOS in a subdirectory of GNU-CC (GNUISH). I 
use frequently XV on a Sun Spark Station 1 and I never had problems, but when I
start it on my computer with -h option, it display the help menu and when I
start it with a GIF-File my Hard disk turns 2 or 3 seconds and the prompt come
back.

My computer is a little 386/25 with copro, 4 Mega rams, Tseng 4000 (1M) running
MS-DOS 5.0 with HIMEM.SYS and no EMM386.SYS. I had the GO32.EXE too... but no
driver who run with i

Categoría: comp.graphics


### 2.2 Transformo a embeddings

In [7]:
print("Modelos de Embeddings disponibles:\n" + "-"*40)
for m in client.models.list():
    if "embed" in m.name:
        print(m.name)

Modelos de Embeddings disponibles:
----------------------------------------
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2


In [8]:
import numpy as np
import time

def obtener_embedding(texto, modelo="gemini-embedding-001"):
    """Llama a la API de Gemini para obtener el embedding de un texto."""
    result = client.models.embed_content(
        model=modelo,
        contents=texto
    )
    return result.embeddings[0].values

def obtener_embeddings_batch(textos, modelo="gemini-embedding-001", batch_size=10, pausa=1.0):
    """Obtiene embeddings de una lista de textos, en lotes para no saturar la API."""
    embeddings = []
    for i in range(0, len(textos), batch_size):
        lote = textos[i:i + batch_size]
        result = client.models.embed_content(
            model=modelo,
            contents=lote
        )
        embeddings.extend([e.values for e in result.embeddings])
        time.sleep(pausa)
    return np.array(embeddings)

embeddings_corpus = obtener_embeddings_batch(documentos)

print(f"Forma de la matriz de embeddings: {embeddings_corpus.shape}")

Forma de la matriz de embeddings: (47, 3072)


### 2.3 Creo una query y hago la búsqueda

In [9]:
from sklearn.metrics.pairwise import cosine_similarity

query = "NASA spacecraft launch and exploration of the solar system"

embedding_query = np.array(obtener_embedding(query)).reshape(1, -1)

similitudes = cosine_similarity(embedding_query, embeddings_corpus)[0]

print(f"Query: {query}")
print(f"\nCantidad de similitudes calculadas: {len(similitudes)}")

Query: NASA spacecraft launch and exploration of the solar system

Cantidad de similitudes calculadas: 47


Obtengo los 5 documentos más similares a mi query

In [10]:
TOP_K = 5

top_indices = np.argsort(similitudes)[::-1][:TOP_K]

print(f"Top {TOP_K} documentos más similares a la query:\n")
print("=" * 80)

for rank, idx in enumerate(top_indices, start=1):
    print(f"\n#{rank} | Categoría: {etiquetas[idx]} | Similitud: {similitudes[idx]:.4f}")
    print("-" * 80)
    print(documentos[idx][:400])
    print("=" * 80)

Top 5 documentos más similares a la query:


#1 | Categoría: sci.space | Similitud: 0.5865
--------------------------------------------------------------------------------
Ah, there's the rub.  And a catch-22 to boot.  For the purposes of a
contest, you'll probably not compete if'n you can't afford the ride to get
there.  And although lower priced delivery systems might be doable, without
demand its doubtful that anyone will develop a new system.  Course, if a
low priced system existed, there might be demand...  

I wonder if there might be some way of structuring a

#2 | Categoría: sci.space | Similitud: 0.5845
--------------------------------------------------------------------------------
No.  The idea was suggested around here during discussions of possible
near-term commercial space activities.  One of the folks involved in those
discussions, a
spacecraft engineer named Preston Carter, passed the suggestion on to 
some entreprenurial types, and Mike Lawson is apparently going ahea

### 2.4 Uso del LLM para responder en base a los documentos recuperados (RAG)

In [12]:
contexto = "\n\n---\n\n".join(
    [f"Documento {i+1} (categoría: {etiquetas[idx]}):\n{documentos[idx]}"
     for i, idx in enumerate(top_indices)]
)

prompt = f"""Respondé la siguiente pregunta utilizando únicamente la información
provista en los documentos a continuación. Si la respuesta no se encuentra en
los documentos, indicalo explícitamente.

Pregunta: {query}

Documentos:
{contexto}

Respuesta:"""

respuesta_rag = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print(respuesta_rag.text)

La información provista en los documentos menciona lo siguiente:

*   NASA está involucrada en el soporte de lanzamiento, cubriendo sus costos incrementales (Documento 2). Se espera que los lanzamientos comerciales se realicen con lanzadores comerciales, no con el Shuttle (Documento 2).
*   Se discute la "exploración lunar tripulada" y "explorar la luna" como objetivos (Documento 1, 5).
*   Se menciona la participación de LLNL en el diseño de sensores ligeros para programas como "Clementine y programas relacionados" (Documento 2), y la recolección de datos de gravedad en la Luna (Documento 3).

Sin embargo, los documentos no proporcionan detalles específicos sobre el lanzamiento de naves espaciales de la NASA ni sobre la exploración del sistema solar más allá de la Luna.
